# Phase 2. Data Understanding

Based on the template, provided by M. Verdegaal via [GitHub](https://github.com/MichaelVerdegaal/crisp-dm-template.git)

In [ ]:
# Loading of technical requierments
# Pandas is used to load and handle the data
import pandas as pd
# Seaborn is used for the majority of graphs
import seaborn as sns
# Just a colour template for visual aspects
sns.set_theme(palette=sns.color_palette("Set2"))
# Basis for plot creatin with python and seaborn
import matplotlib.pyplot as plt
#Needed to render plots within the Notebook
%matplotlib inline 

# 2.1 Collect initial data

## 2.1.1 Task
*Acquire the data (or access to the data) listed in the project resources. This initial collection includes data loading, if necessary for data understanding.*

The Data is pulled from the original source and saved as a pandas dataframe to prepare it for the following steps.

In [ ]:
# Loading the Dataset from the original source
url = "https://prod-dcd-datasets-public-files-eu-west-1.s3.eu-west-1.amazonaws.com/5ba56a64-44fc-42ad-92ea-9cb34550e09c"
data = pd.read_csv(url)

## 2.1.1 Output

### 2.1.1.1 Initial data collection report

*List the dataset(s) acquired, together with their locations, the methods used to acquire them, and any problems encountered (and how those problems were resolved).*

In this project the dataset "Cardiovascular_Disease_Dataset" provided by Doppala, Bhanu Prakash and Bhattacharyya, Debnath in 2021 is used. Which can be found here: DOI 10.17632/dzz48mvjht.1. Additionaly they provide a Description Sheet for the Dataset. A full report can be found in the Projektpaper.

# 2.2 Describe data

## 2.2.1 Task

*Examine the “gross” or “surface” properties of the acquired data and report on the results.*

To achieve an overview, the function pd.head() is used, which prints the Attributes, along with the five first entrys.  
Aswell as the function pd.info() which returns the technial information on the Attributes.

In [ ]:
#  Printing the head of the dataset to provide visualisation
data.head()

In [ ]:
# Printing the datatypes aswell in a compact format
data.info()


## 2.2.2 Output

### 2.2.2.1 Data description report

*Describe the data that has been acquired, including the format of the data, the quantity of data (for example, the number of records and fields in each table), the identities of the fields, and any other surface features which have been discovered.*

The dataset on Cardiovascular Disese contains one thousand entrys, each composed of fourteen features.  
There is limited information on the circumstances of collection, what is provided is that the data has been collected in a multispeciality hospital in India.

There are the Patient Identification Number, acting as a uniqe identifier while maintaining privacy of individuals, the numerical Age of the patients, the Gender, only expressed in a binary format, the Chest Pain Type, differentiated in four nominal Types, some hematological Data, being the resting blood pressure - mm HG, the serum cholesterol - mg/dl, and the fasting blood sugar - recorded in binary as more or less then 120 mg/dl. 

Also included are the resting electrocardiogramm results, using three nominal categories, the maximum heart rate - numerical, the oocurence of exercise induced angina, recored in binary, the "oldpeak" which describes a specfic area of the ST-Wave on a cardiogramm - numeric, the slope of that area, - nominal, up, flat or down, and the number of visible major vessels - numeric. Lastly the Absence or Presence of Heart Disese or Target, is recorded as yes or no.

Regarding the technical aspects, the data has seemingly already been prepared for computational purposes, as all values are represented as a workable int or float.

# 2.3 Explore data

## 2.3.1 Task

*This task addresses data mining questions using querying, visualization, and reporting techniques. These include distribution of key attributes (for example, the target attribute of a prediction task) relationships between pairs or small numbers of attributes, results of simple aggregations, properties of significant sub-populations, and simple statistical analyses.*

To visualise all of the included Data, a histogramm is generated for all attributes (excluding patient ID).  
This is only gives a very unspecific overview, but provides a good starting point.

In [ ]:
# Dropping the Patient ID
data_noID = data.drop(columns=['patientid'])
data_noID.hist(figsize=(20,12))

To advance the knowledge concerning the patients, we can look at the Age and Gender Distribution.  
Note: This visualisation is normalized and does not show participant count for each gender.

In [ ]:
age_gender_df = data[['age', 'gender']].copy()
age_gender_df['gender'] = age_gender_df['gender'].map({0: 'Female', 1: 'Male'})
sns.violinplot(data=age_gender_df, y="age", hue='gender', split = True)
plt.title("Age Distribution by Gender")

To represent the difference in group size between men and women, we can show the absolute values.

In [ ]:
sns.violinplot(data=age_gender_df, y="age", x='gender', density_norm='count', split = True)
plt.title("Absolute Age Distribution by Gender")

As the age appears to be well distributed, we can see how the target value behaves in dependence of the age.

In [ ]:
age_target_df = data[['age', 'target']].copy()
age_target_df['target'] = age_target_df['target'].map({0: 'No Heart Disease', 1: 'Heart Disease'})
sns.violinplot(data=age_target_df, y="age", hue='target', split = True)
plt.title('Distribution of Age by Target')

The plot shows, heart diseases in this study, appear to be more prevalent in people around the age of 50.

This concludes the overview of the participants for now.  

To increase understanding of the complete dataset, a correlation Matrix will be created.

In [ ]:
#Creating the correlation matrix
corr = data_noID.corr()
#Creating the matplot figure
f, ax = plt.subplots(figsize=(20,12))
#Custom Colourmap
#cmap= sns.diverging_palette(140,870,as_cmap = True)
#cmap = sns.light_palette("seagreen", as_cmap=True)
cmap = sns.color_palette("icefire", as_cmap=True)
#Drawing the Matrix
data_heatmap = sns.heatmap(corr, cmap=cmap, square = True, ax=ax, annot = True, linewidth=0.1, annot_kws={"size":16})
plt.title('Correlation Matrix', y=1.05,size=15)


Using this matrix, we can see that the previously evaluated pairs, do not exhibit a meaningful correlation.  
We can however also see some strong correlations, especially  **chestpain - target** and **slope - target**.

In [ ]:
target_chestpain_df = data_noID[['chestpain', 'target']].copy()
target_chestpain_df['target'] = target_chestpain_df['target'].map({0: 'No Heart Disease', 1: 'Heart Disease'})
target_chestpain_df['chestpain'] = target_chestpain_df['chestpain'].map({0: 'Typical Angina', 1: 'Atypical Angina', 2: 'Non-Angina', 3:'Asymptomatic'})
sns.histplot(data=target_chestpain_df, x="chestpain", hue = 'target', multiple = 'fill')
plt.title('Distribution of Chestpain in correlation to Target')

We can see that a **Typical Angina** does not appear to indicate a Heart Disease.

In [ ]:
sns.countplot(data=target_chestpain_df, x="chestpain", hue = 'target')
plt.title('Absolute Distribution of Chestpain in correlation to Target')

Looking at the absolute numbers, patients experiencing a **Typical Angina** were the second least likely, after asymptomatic patients, to have a Heart Disease.

Proceeding to **slope - target**  
The appearing invalid value will be explained at a later point

In [ ]:
target_slope_df = data_noID[['slope', 'target']].copy()
target_slope_df['target'] = target_slope_df['target'].map({0: 'No Heart Disease', 1: 'Heart Disease'})
target_slope_df['slope'] = target_slope_df['slope'].map({0:'Invalid', 1: 'Upslope', 2: 'Flat', 3:'Downslope'})
sns.violinplot(data=target_slope_df, x="slope", hue = 'target', split = True)
plt.title('Distribution of slope in correlation to Target')

From the visualisation, it can be concluded, that a **Downslope** or a **Flat Slope**, are major indicators for a **heart disease**.

## 2.3.2 Output

### 2.3.2.1 Data exploration report

*Describe results of this task, including first findings or initial hypothesis and their impact on the remainder of the project.*

In this section, weve taken a look at all attributes in our dataset and further explored the patients whose data has been recorded.  
These patients are evenly distributet between the ages of 20 and 80 and predominantly male, with less then 25% women.  
Weve seen that heart diseases appears to be more prevalent in participants around the age of 50.  

The whole of the dataset has been further explored using a correlation Matrix,  
that pointed towards strong correlations between  **chestpain - target** and **slope - target**.  
Further investigation of these attributes showed that a **Typical Angina** does not appear to indicate a Heart Disease.  
Looking at the absolute numbers, patients experiencing a **Typical Angina** were the second least likely, after asymptomatic patients, to have a Heart Disease.  
Proceeding to **slope - target**, the visualisation, indicates that a **Downslope** or a **Flat Slope**, are major indicators for a **heart disease**.  
*The appearing invalid value will be explained at a later point*  

# 2.4 Verify data quality

## 2.4.1 Task

*Examine the quality of the data, addressing questions such as: Is the data complete (does it
cover all the cases required)? Is it correct, or does it contain errors and, if there are
errors, how common are they?*

As weve noted before, there already have been some supsicious entrys, but before adressing that, we will check the dataset for duplicates.  
Once with the ID, and once without.

In [ ]:
#Checking for duplicate entries
duplicates = data[data.duplicated()]
print("Number of Duplicates (with ID): ",  duplicates.shape[0])
duplicates = data_noID[data.duplicated()]
print("Number of Duplicates(without ID): ", duplicates.shape[0])

As both of these return 0 we can be reasonably certain that no entrys have been duplicated.

Another useful tool to examine the data is the data.describe() function that is build into pandas.

In [ ]:
# A table with various metrics about the dataset
data.describe()

All Datasets appear complete, with a thousend entrys for every attribute.  
There are however invalid entrys that collide with the specifications from the Description Sheet.  
In particular the serumcholestrol and the slope contain zero values that are not valid via the defintion.



## 2.4.2 Output

### 2.4.2.1 Data quality verification report

*List the results of the data quality verification; if quality problems exist, list possible
solutions.*


Since the patient ID does not hold any information itself, we will start with the age, ranging from 20 to 80 with a mean age of 49 which appears to be higher then Indias average age of 30,9 in 2021, accoarding to [GlobalData](https://www.globaldata.com/data-insights/macroeconomic/the-mean-age-in-india-139012/), but may be explained since only hospital patients were participating, whose average age might reasonably be higher.

Gender, which has only been recorded in binary, reveals that more then 75% of the particpents were male. This might negativly impact the significance of any findings concerning female patientes. The type of chestpain seems well distributed between the patients. With the least patients being asymptomatic and more then 75% experiencing either anginal nor non-anginal pain. The hematological Data also seems well distributed, without having medical background knowledge of values that can be expected. 

Even though they are well distributed,**the min value of 0**, that occours for serumcholestrol **suggests missing values** that have been replaced by zero.
This observation might have reaching implications for the whole dataset, if this has happend at other places.

More then 50% of particpants are reported to have a fasting blood sugar, lower then 120 mg/dl, which seems to be possible.
The results for the resting electrocardiogramm appear to skew towards categorie one, which represens atypical behaviour of the ST-Wave, but this might be accoarding to expectations, considering that the datasets is dealing with patients suffering from diseses impacting the heart.
The maximum heart rate, the exercise induced angina and oldpeak appear to reasonably fit the possible values.
The slope however provides the next **indication** that **missing values** might **have been replaced by zero**, since the categorie does not allow for zero originaly.
Lastly the number of major vessels and the target variable do not give immediate reason for concern.

All plots will be repeated, excluding entrys containing zeros in serumcolestrol or slope.

In [ ]:
# Dropping the zero values
data_noZeros = data[(data['slope'] != 0) & (data['serumcholestrol'] != 0)]
data_noZeros.describe()

In [ ]:
# Dropping the Patient ID
data_noID = data_noZeros.drop(columns=['patientid'])
data_noID.hist(figsize=(20,12))

In [ ]:
age_gender_df = data_noZeros[['age', 'gender']].copy()
age_gender_df['gender'] = age_gender_df['gender'].map({0: 'Female', 1: 'Male'})
sns.violinplot(data=age_gender_df, y="age", hue='gender', split = True)
plt.title("Age Distribution by Gender")

In [ ]:
sns.violinplot(data=age_gender_df, y="age", x='gender', density_norm='count', split = True)
plt.title("Absolute Age Distribution by Gender")

In [ ]:
age_target_df = data_noZeros[['age', 'target']].copy()
age_target_df['target'] = age_target_df['target'].map({0: 'No Heart Disease', 1: 'Heart Disease'})
sns.violinplot(data=age_target_df, y="age", hue='target', split = True)
plt.title('Distribution of Age by Target')

This appears to actually reverse the previous observations.

In [ ]:
#Creating the correlation matrix
data_noID = data_noZeros.drop(columns=['patientid'])
corr = data_noID.corr()
#Creating the matplot figure
f, ax = plt.subplots(figsize=(20,12))
#Custom Colourmap
#cmap= sns.diverging_palette(140,870,as_cmap = True)
#cmap = sns.light_palette("seagreen", as_cmap=True)
cmap = sns.color_palette("icefire", as_cmap=True)
#Drawing the Matrix
data_heatmap = sns.heatmap(corr, cmap=cmap, square = True, ax=ax, annot = True, linewidth=0.1, annot_kws={"size":16})
plt.title('Correlation Matrix', y=1.05,size=15)

The correlations between chestpain and targer, and between slope and targer appear weakend.

In [ ]:
target_chestpain_df = data_noID[['chestpain', 'target']].copy()
target_chestpain_df['target'] = target_chestpain_df['target'].map({0: 'No Heart Disease', 1: 'Heart Disease'})
target_chestpain_df['chestpain'] = target_chestpain_df['chestpain'].map({0: 'Typical Angina', 1: 'Atypical Angina', 2: 'Non-Angina', 3:'Asymptomatic'})
sns.histplot(data=target_chestpain_df, x="chestpain", hue = 'target', multiple = 'fill')
plt.title('Distribution of Chestpain in correlation to Target')

In [ ]:
sns.countplot(data=target_chestpain_df, x="chestpain", hue = 'target')
plt.title('Absolute Distribution of Chestpain in correlation to Target')

In [ ]:
target_slope_df = data_noID[['slope', 'target']].copy()
target_slope_df['target'] = target_slope_df['target'].map({0: 'No Heart Disease', 1: 'Heart Disease'})
target_slope_df['slope'] = target_slope_df['slope'].map({0:'Invalid', 1: 'Upslope', 2: 'Flat', 3:'Downslope'})
sns.violinplot(data=target_slope_df, x="slope", hue = 'target', split = True)
plt.title('Distribution of slope in correlation to Target')